# Untrained random policy

Joint angles → small neural network → 18 bounded offsets. Random weights stay fixed; action noise changes each step. No learning yet.

[Earlier coordinated visual aid](https://github.com/haidmoham/spider/blob/d6238be2bbedf224322e07539e45c68a49525830/lab/notebooks/03_coordinated_baseline.ipynb).

In [ ]:
from pathlib import Path
import sys
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "spider" / "learning.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
import mujoco
import numpy as np
import pandas as pd
from spider.learning import LearningSimulation, record_policy, plot_recordings
from spider.simulation import neutral_targets
sim = LearningSimulation()
print("Actuator order:", [sim.model.actuator(i).name for i in range(sim.model.nu)])
print("Timestep:", sim.model.opt.timestep, "s")

In [ ]:
import numpy as np

def make_random_policy(weight_seed=7, action_seed=11):
    weight_rng = np.random.default_rng(weight_seed)
    action_rng = np.random.default_rng(action_seed)

    # 18 inputs → 32 hidden units → 18 outputs. Initialize once.
    w1 = weight_rng.normal(0.0, 1.0 / np.sqrt(18), size=(18, 32))
    b1 = np.zeros(32)
    w2 = weight_rng.normal(0.0, 0.01 / np.sqrt(32), size=(32, 18))
    b2 = np.zeros(18)

    max_offset_rad = 0.05  # ±2.9 degrees from neutral.
    exploration_std = 0.20  # Noise before tanh.

    def random_policy(observation):
        x = np.asarray(observation.joint_positions, dtype=float) / 1.0
        hidden = np.tanh(x @ w1 + b1)
        mean = hidden @ w2 + b2

        # Fresh noise each call; tanh bounds the final offsets.
        noise = action_rng.normal(0.0, exploration_std, size=18)
        offsets = max_offset_rad * np.tanh(mean + noise)
        return offsets

    return random_policy

random_policy = make_random_policy()

In [ ]:
random_recording = record_policy(
    make_random_policy(weight_seed=7, action_seed=11),
    label="Untrained random | seeds 7/11 | bound 0.05 rad | noise 0.20",
    action_count=100, physics_steps=10,
)
times = np.asarray(random_recording.replay.times)
expected = np.arange(101) * 10 * random_recording.replay.model.opt.timestep
assert np.allclose(times, expected, rtol=0, atol=1e-9), "Simulation clock reset: inspect invalid trial."
assert np.isfinite(random_recording.replay.states).all()
print(f"Recorded {times[-1]:.2f} seconds; {len(random_recording.offsets)} actions.")
fig, axes = plot_recordings(random_recording, actuator=0)

In [ ]:
viewer = random_recording.watch(REPO_ROOT / "telemetry" / "random-policy-replay", speed=1.0)